# Agent that has multiple tools available

In [17]:
from dotenv import load_dotenv
load_dotenv("../../.env")


from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

from IPython.display import Image, display

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_openai_messages

from qdrant_client.http.models import MatchValue, FieldCondition, Filter
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, RrfQuery, Rrf

import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree
import instructor
from pydantic import BaseModel, Field

from jinja2 import Template

from typing import Annotated, List, Any
from operator import add

from sentence_transformers import CrossEncoder

# persistence in multi turn conversations
from langgraph.checkpoint.postgres import PostgresSaver
import os

# creating new collection
import pandas as pd
import tiktoken

### Creating new Rating Qdrant collection

In [18]:
qdrant_client = QdrantClient(url="localhost:6333")

In [19]:
COLLECTION_NAME = "Amazon-reviews-collection-hybrid-search"

try:
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE),
        },
        sparse_vectors_config={
            "bm25": SparseVectorParams(modifier=Modifier.IDF),
        }
    )
except Exception as e:
    print(f"Collection probably already exists! Here is the error message:\n{e}")

Collection probably already exists! Here is the error message:
Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `Amazon-reviews-collection-hybrid-search` already exists!"},"time":0.005195125}'


In [20]:
qdrant_client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=6, status=<UpdateStatus.COMPLETED: 'completed'>)

In [21]:
### EMBEDDINGS ###
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend(
            [embedding.embedding for embedding in response.data]
        )

        print(f"Batch #{counter}: {counter * batch_size} / {len(text_list)}")
        counter += 1

    return all_embeddings

In [22]:
# read the ratings dataset
df_reviews = pd.read_json("../../data/Electornics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [23]:
df_reviews.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Camera & Photo,8x8ft Pink Rose Flower Theme Photography Backd...,4.5,358,"[ʚɞSize：8x8FT 2.4 m wide by 2.4 m tall,Item se...",[Details 1. The item color as the picture show...,38.90,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Avezano Brand Video', 'url': 'http...",XLL,"[Electronics, Camera & Photo, Lighting & Studi...",{'Package Dimensions': '10.94 x 9.57 x 1.97 in...,B08XQ8PS2J,NaN
1,Cell Phones & Accessories,Spigen Tempered Glass Screen Protector [GlasTR...,4.3,921,[Tempered glass durability rated at 9H hardnes...,[],32.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Gal Tab S8 Series Installation Gui...,Spigen,"[Electronics, Computers & Accessories, Tablet ...","{'Product Dimensions': '12.71""L x 8.07""W', 'It...",B09P5NQH7Q,NaN
2,Amazon Home,COLSUR【2023 Newest Bluetooth Speaker with Digi...,4.3,580,[【6 IN 1 Multifunctional Alarm Clock Design】- ...,[],56.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Digital Alarm Clock Radio with Blu...,COLSUR,"[Electronics, Home Audio, Compact Radios & Ste...","{'Brand': 'COLSUR', 'Color': 'Wooden', 'Displa...",B0B4JVPWMJ,NaN
3,Computers,Beelink Mini pc 10 Cores 12th Generation Intel...,4.4,263,[【12th Gen Intel i5 Ultra High-efficiency】Beel...,[],449.00,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'SEi12 i5-1235U 16+500GB', 'url': '...",Beelink,"[Electronics, Computers & Accessories, Compute...","{'Processor': '4.4 GHz core_i5', 'RAM': '16 GB...",B0C69CCSJW,NaN
4,All Electronics,ZOVITS U-Shuxian Wireless Earbuds Blutooth 5.3...,4.5,452,"[LED power display charging case, deep BASS, S...",[ZOVITS U-Shuxian Wireless Earbuds Lightweight],22.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '30 Hrs of Playtime for this Price?...,ZOVITS,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '4.29 x 3.98 x 1.54 inc...,B0BFR3QPWR,NaN


In [24]:
len(df_reviews)

1000

In [25]:
# preprocess title and features
def preprocess_reviews_data(row):
    return f"{row['title']} {row["description"]}"

In [28]:
def count_tokens(text: str):
    encoding = tiktoken.encoding_for_model("text-embedding-3-small")
    return len(encoding.encode(text))

def count_tokens_row(row):
    return count_tokens(row["preprocessed_content"])

In [29]:
count_tokens("Hi")

1

In [30]:
df_reviews["preprocessed_content"] = df_reviews.apply(preprocess_reviews_data, axis=1)
df_reviews["token_count"] = df_reviews.apply(count_tokens_row, axis=1)

In [31]:
df_reviews.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,preprocessed_content,token_count
0,Camera & Photo,8x8ft Pink Rose Flower Theme Photography Backd...,4.5,358,"[ʚɞSize：8x8FT 2.4 m wide by 2.4 m tall,Item se...",[Details 1. The item color as the picture show...,38.90,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Avezano Brand Video', 'url': 'http...",XLL,"[Electronics, Camera & Photo, Lighting & Studi...",{'Package Dimensions': '10.94 x 9.57 x 1.97 in...,B08XQ8PS2J,NaN,8x8ft Pink Rose Flower Theme Photography Backd...,274
1,Cell Phones & Accessories,Spigen Tempered Glass Screen Protector [GlasTR...,4.3,921,[Tempered glass durability rated at 9H hardnes...,[],32.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Gal Tab S8 Series Installation Gui...,Spigen,"[Electronics, Computers & Accessories, Tablet ...","{'Product Dimensions': '12.71""L x 8.07""W', 'It...",B09P5NQH7Q,NaN,Spigen Tempered Glass Screen Protector [GlasTR...,32
2,Amazon Home,COLSUR【2023 Newest Bluetooth Speaker with Digi...,4.3,580,[【6 IN 1 Multifunctional Alarm Clock Design】- ...,[],56.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Digital Alarm Clock Radio with Blu...,COLSUR,"[Electronics, Home Audio, Compact Radios & Ste...","{'Brand': 'COLSUR', 'Color': 'Wooden', 'Displa...",B0B4JVPWMJ,NaN,COLSUR【2023 Newest Bluetooth Speaker with Digi...,45
3,Computers,Beelink Mini pc 10 Cores 12th Generation Intel...,4.4,263,[【12th Gen Intel i5 Ultra High-efficiency】Beel...,[],449.00,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'SEi12 i5-1235U 16+500GB', 'url': '...",Beelink,"[Electronics, Computers & Accessories, Compute...","{'Processor': '4.4 GHz core_i5', 'RAM': '16 GB...",B0C69CCSJW,NaN,Beelink Mini pc 10 Cores 12th Generation Intel...,77
4,All Electronics,ZOVITS U-Shuxian Wireless Earbuds Blutooth 5.3...,4.5,452,"[LED power display charging case, deep BASS, S...",[ZOVITS U-Shuxian Wireless Earbuds Lightweight],22.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '30 Hrs of Playtime for this Price?...,ZOVITS,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '4.29 x 3.98 x 1.54 inc...,B0BFR3QPWR,NaN,ZOVITS U-Shuxian Wireless Earbuds Blutooth 5.3...,36


In [32]:
# filter out very long reviews
df_reviews = df_reviews[df_reviews["token_count"] < 8192]

In [33]:
len(df_reviews)

1000

In [36]:
# total token count to embed
total_tokens = df_reviews["token_count"].sum()
total_tokens

np.int64(92775)

In [38]:
total_cost = total_tokens * 0.00000002 #price per token for text-embedding-3-small in USD
print("Total cost to embed in USD:", total_cost)

Total cost to embed in USD: 0.0018555


In [39]:
data_to_embed = df_reviews[["preprocessed_content", "parent_asin"]].to_dict(orient="records")

In [40]:
data_to_embed

[{'preprocessed_content': "8x8ft Pink Rose Flower Theme Photography Backdrops 3D Pink Floral Newborn Kids Wedding Bridal Shower Photo Background Mother's Day Anniversary Ceremony Couple Photo Props Cake Table Banner ['Details 1. The item color as the picture shown. 2. Made of vinyl cloth. Which is lightweight, thin, easy to carry, durable and wrinkle . 3. May have some creases, but it can be removed. 4. Cannot be washed. 5. Can be folded, easy to place. 6. Non-reflective. 7. The package just have one piece of backdrop. How to remove wrinkles (1)Hang naturally for a few days, no problem. (2) Heat it with a steam iron on the back of item, if necessary, please iron the back surface with steam iron but not dry iron. Then it will be smooth again. Warm-tip Because of the difference between the monitor and the light, here is normal that a slight difference in the color. The picture is clear, realistic, stereo sense is strong. Applications range: It can be used at many situations, such as gett

In [41]:
text_to_embed = [row["preprocessed_content"] for row in data_to_embed]
text_to_embed

["8x8ft Pink Rose Flower Theme Photography Backdrops 3D Pink Floral Newborn Kids Wedding Bridal Shower Photo Background Mother's Day Anniversary Ceremony Couple Photo Props Cake Table Banner ['Details 1. The item color as the picture shown. 2. Made of vinyl cloth. Which is lightweight, thin, easy to carry, durable and wrinkle . 3. May have some creases, but it can be removed. 4. Cannot be washed. 5. Can be folded, easy to place. 6. Non-reflective. 7. The package just have one piece of backdrop. How to remove wrinkles (1)Hang naturally for a few days, no problem. (2) Heat it with a steam iron on the back of item, if necessary, please iron the back surface with steam iron but not dry iron. Then it will be smooth again. Warm-tip Because of the difference between the monitor and the light, here is normal that a slight difference in the color. The picture is clear, realistic, stereo sense is strong. Applications range: It can be used at many situations, such as getting together occasion, we

In [ ]:
#### Performing Embedding ####
embeddings = get_embeddings_batch(text_to_embed, batch_size=200)

Batch #1: 200 / 1000
Batch #2: 400 / 1000
Batch #3: 600 / 1000
Batch #4: 800 / 1000
Batch #5: 1000 / 1000


In [43]:
pointstructs = []
counter = 1

for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=counter,
            vector={
                "text-embedding-3-small": embedding
            },
            payload=data
        )
    )
    counter += 1

In [44]:
pointstructs[0].vector

{'text-embedding-3-small': [0.03125,
  0.027130126953125,
  -0.01345062255859375,
  -0.012481689453125,
  -0.03729248046875,
  -0.0423583984375,
  0.014495849609375,
  -0.0188751220703125,
  -0.01528167724609375,
  0.021575927734375,
  -0.0130157470703125,
  0.033782958984375,
  -0.01280975341796875,
  -0.031280517578125,
  -0.0234222412109375,
  0.046051025390625,
  0.032684326171875,
  0.047943115234375,
  -0.0002968311309814453,
  0.01134490966796875,
  0.038055419921875,
  -0.005641937255859375,
  -0.025421142578125,
  -0.0350341796875,
  0.036590576171875,
  0.01433563232421875,
  0.07196044921875,
  -0.0330810546875,
  0.02685546875,
  -0.0003001689910888672,
  -0.034576416015625,
  -0.035491943359375,
  0.037139892578125,
  -0.07183837890625,
  0.0202178955078125,
  0.0023193359375,
  -0.0233154296875,
  -0.01317596435546875,
  -0.0194244384765625,
  0.01073455810546875,
  0.053436279296875,
  0.017974853515625,
  0.0208740234375,
  -0.045562744140625,
  -0.02239990234375,
  0.0

In [47]:
# upload to vector db
def upload_to_vector_db(pointstructs, batch_size=100):
    counter = 1
    for i in range(0,len(pointstructs), batch_size):
        batch = pointstructs[i:i+batch_size]    

        qdrant_client.upsert(
            collection_name=COLLECTION_NAME,
            points=batch,
            wait=True
        )
        print(f"Processed {counter * batch_size} / {len(pointstructs)}")
        counter += 1

In [48]:
upload_to_vector_db(pointstructs)

Processed 100 / 1000
Processed 200 / 1000
Processed 300 / 1000
Processed 400 / 1000
Processed 500 / 1000
Processed 600 / 1000
Processed 700 / 1000
Processed 800 / 1000
Processed 900 / 1000
Processed 1000 / 1000


### Searching a prefiltered set of product ids

In [ ]:
from qdrant_client.http.models import FusionQuery
from qdrant_client.http.models import MatchAny
def prefiltered_reviews(query: str, parent_asins: list[str], k: int = 10) -> str:
    query_embedding = get_embeddings_batch(query)[0]

    #hybrid retrieval
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin", 
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            ),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    return results

In [56]:
reviews = prefiltered_reviews("itunes", ["B0BKVR14JN", "B09KQP2H7N", "B0C996WY16"])
reviews.points

[ScoredPoint(id=216, version=10, score=0.5, payload={'preprocessed_content': 'MP3 Player with 32GB TF Card,Built-in HD Speaker,Portable HiFi Music Player with Video/Voice Recorder/FM Radio/Photo Viewer/E-Book Player for Kids [\'Xidehuy Music Playeris fashion and delicate in design.you can take it anywhere with your favorite songs, videos, e-books and important files. It is an economical gift for yourself, your family and your friends. Warm Notice:Large capacity, a 32 GB memory card is included. (Support up to 128 GB Card for Maximum). ●Screen Size: 1.8" High Resolution Real Color LCD Screen ●Built-in Li-ion battery, can be charged via USB cable to the PC or wall socket adapter ●Charging time: about 1 hours. Warm reminder: it is recommended to turn on the player before charging. ●Product weight (package not included): about 0.029kg ●Playing time: about 8-12hours ●MP3 Size: 3.5 x 1.5 x 0.3 in ●Support music playback format: MP3、WMA、OGG、APE、FLAC、WAV、AAC-LC、ACELP ●Video format: This player

In [63]:
from qdrant_client.http.models import FusionQuery
from qdrant_client.http.models import MatchAny
def retrieve_prefiltered_reviews(query: str, parent_asins: list[str], k: int = 10) -> str:
    query_embedding = get_embeddings_batch(query)[0]

    #hybrid retrieval
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin", 
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            ),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )


    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_content"])
        similarity_scores.append(result.score)
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores
    }


def format_context(context: dict):
    # formatting the retrieved & reranked context
    formatted_context = ""

    for id, review in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, User Review: {review}\n"

    return formatted_context

In [64]:
def retrieve_formatted_context(query: str, parent_asins: list[str], k: int = 10) -> str:
    """Get the top k user reviews for a list of prefiltered items, based on the provided parent asin.

    Arg:
        query: The search string that is used to perform search for user reviews
        parent_asins: The list of item IDs to prefilter the reviews
        k: Top k items that will be returned by the tool

    Returns:
        A formatted string that contains the k user reviews for the prefiltered product ids that best fit the searched query
    """
    retrieved_context = retrieve_prefiltered_reviews(query=query, parent_asins=parent_asins, k=k)
    formatted_context = format_context(retrieved_context)
    return formatted_context

In [65]:
result = retrieve_formatted_context("itunes", ["B0BKVR14JN", "B09KQP2H7N", "B0C996WY16"])

In [67]:
print(result)

- ID: B0BKVR14JN, User Review: MP3 Player with 32GB TF Card,Built-in HD Speaker,Portable HiFi Music Player with Video/Voice Recorder/FM Radio/Photo Viewer/E-Book Player for Kids ['Xidehuy Music Playeris fashion and delicate in design.you can take it anywhere with your favorite songs, videos, e-books and important files. It is an economical gift for yourself, your family and your friends. Warm Notice:Large capacity, a 32 GB memory card is included. (Support up to 128 GB Card for Maximum). ●Screen Size: 1.8" High Resolution Real Color LCD Screen ●Built-in Li-ion battery, can be charged via USB cable to the PC or wall socket adapter ●Charging time: about 1 hours. Warm reminder: it is recommended to turn on the player before charging. ●Product weight (package not included): about 0.029kg ●Playing time: about 8-12hours ●MP3 Size: 3.5 x 1.5 x 0.3 in ●Support music playback format: MP3、WMA、OGG、APE、FLAC、WAV、AAC-LC、ACELP ●Video format: This player recognizes AMV and AVI video files(It needs Act